# Audit qualité du jalon 1

## tl;dr

Le dataset versionné contient **36 423 matchs**. L'audit reproductible identifie
**24 segments** `SUSPECT_ZERO`, soit **7 936 valeurs** conservées mais exclues par
défaut des modèles concernés. Elles se concentrent sur F2, N1 et P1 en 2015-16 et
2016-17. Aucun doublon `match_id`, score final manquant ou match futur n'est détecté.

## Context & Methods

Objectif : vérifier le grain match, l'unicité, la complétude temporelle et les
zéros historiquement imputés. Le fichier source est `data/matches.parquet`.

### Key Assumptions

- grain attendu : un match terminé par ligne ;
- `match_id` est la clé legacy à auditer, pas la future identité interne ;
- un couple de statistiques intégralement nul sur une ligue-saison est suspect,
  jamais corrigé automatiquement.

## Data

### 1. Charger et profiler

In [1]:
from datetime import UTC, datetime
from pathlib import Path
import pandas as pd
from robin.quality.checks import run_match_checks
from robin.quality.zero_audit import audit_suspect_zeros

root = Path.cwd()
matches = pd.read_parquet(root / "data" / "matches.parquet")
profile = {
    "rows": len(matches),
    "columns": len(matches.columns),
    "date_min": str(matches["date"].min()),
    "date_max": str(matches["date"].max()),
    "leagues": matches["league"].nunique(),
    "seasons": matches["season"].nunique(),
    "duplicate_match_id": int(matches["match_id"].duplicated().sum()),
}
pd.Series(profile, name="value")

rows                                36423
columns                                27
date_min              2015-07-31 00:00:00
date_max              2026-05-24 00:00:00
leagues                                 9
seasons                                11
duplicate_match_id                      0
Name: value, dtype: object

## Results

### 2. Détecter les zéros suspects

In [2]:
audit = audit_suspect_zeros(matches, provider="football-data.co.uk")
audit_frame = pd.DataFrame([row.model_dump(mode="json") for row in audit])
suspect = audit_frame[audit_frame["quality_status"] == "SUSPECT_ZERO"]
summary = {
    "segments_audited": len(audit_frame),
    "suspect_segments": len(suspect),
    "suspect_values": int(suspect["zeros"].sum()),
}
pd.Series(summary, name="value")

segments_audited     792
suspect_segments      24
suspect_values      7936
Name: value, dtype: int64

In [3]:
suspect.groupby(
    ["competition", "season"], as_index=False
).agg(columns=("column", "count"), suspect_values=("zeros", "sum"))

,competition,season,columns,suspect_values
0,F2,2015-16,4,1520
1,F2,2016-17,4,1520
2,N1,2015-16,4,1224
3,N1,2016-17,4,1224
4,P1,2015-16,4,1224
5,P1,2016-17,4,1224


### 3. Exécuter les contrôles de confiance

In [4]:
checks = run_match_checks(
    matches,
    audit,
    as_of_time=datetime(2026, 7, 24, 23, 59, tzinfo=UTC),
)
pd.DataFrame([check.model_dump(mode="json") for check in checks])[
    ["check_name", "status", "severity", "observed_value", "affected_rows"]
]

,check_name,status,severity,observed_value,affected_rows
0,required_schema_completeness,PASSED,CRITICAL,"colonnes absentes=[], lignes incomplètes=0",0
1,match_id_uniqueness,PASSED,CRITICAL,0 identifiants dupliqués,0
2,fixture_business_uniqueness,PASSED,CRITICAL,0 fixtures métier dupliqués,0
3,final_score_completeness,PASSED,CRITICAL,0 scores incomplets,0
4,score_domain_validity,PASSED,HIGH,"0 scores hors domaine [0, 20]",0
5,future_data,PASSED,CRITICAL,0 lignes futures,0
6,dataset_freshness,PASSED,HIGH,dernière date il y a 61 jours,0
7,suspect_zero_segments,WARNING,HIGH,"24 segments, 7936 valeurs suspectes",7936
8,internal_identity_coverage,WARNING,HIGH,dataset legacy non migré vers les UUID internes,36423
9,source_conflict_observability,WARNING,MEDIUM,provenance source absente du dataset legacy,36423


## Takeaways

- Le grain historique est unique sur `match_id`, mais cette clé reste à remplacer
  par l'identité interne stable du jalon 1.
- Les 7 936 zéros suspects ne sont ni supprimés ni remplacés : leur statut qualité
  les exclut des features concernées.
- Les rapports Vague 2/Vague 2B antérieurs utilisant ces segments restent
  `UNVERIFIED`.
- La preuve détaillée est exportée dans `docs/data-quality/`.